# تدريب نموذج ترجمة المانهوا (مجاني)

هذا الدفتر يدرّب نموذج **Qwen** على بيانات الترجمة المجمّعة في مستودعك على Hugging Face، عبر **Unsloth** — بدون أي رصيد مدفوع.

### قبل البدء تأكد من:
1. رفعتَ بيانات في مستودع البيانات (مثل `kzome/manhua-ar-training`).
2. لديك توكن Hugging Face بصلاحية كتابة من [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
3. هذا الدفتر يعمل على GPU: **Runtime → Change runtime type → T4 GPU**.

---

In [ ]:
# ═══════════════════════ إعداداتك هنا ═══════════════════════
HF_TOKEN = "hf_ضع_توكنك_هنا"        # توكن Hugging Face
DATASET_REPO = "kzome/manhua-ar-training"   # مستودع بيانات التدريب
OUTPUT_REPO = "kzome/manhua-ar-model"       # مستودع النموذج الناتج (يُنشأ تلقائياً)
MODEL_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"  # النموذج الأساسي
EPOCHS = 3
MAX_SEQ_LEN = 1024
LEARNING_RATE = 2e-4
# ════════════════════════════════════════════════════════════

assert HF_TOKEN.startswith("hf_"), "ضع توكن Hugging Face في HF_TOKEN"
assert DATASET_REPO, "ضع اسم مستودع البيانات في DATASET_REPO"
assert OUTPUT_REPO, "ضع اسم مستودع الناتج في OUTPUT_REPO"
print("✅ الإعدادات مكتملة")

## 1) تثبيت المكتبات

In [ ]:
!pip install --upgrade --quiet pip
# 1) إجبار إصدارات متوافقة مع unsloth أولاً (torch<2.12, transformers<=5.5, datasets<4.4, trl<=0.24)
!pip install --quiet "torch==2.11.0" "torchvision==0.26.0" "transformers==4.57.1" "datasets==4.3.0" "trl==0.22.2" "peft>=0.18.0" "bitsandbytes" "xformers" "accelerate"
# 2) ثم تثبيت unsloth (سيرى الإصدارات المتوافقة ولن يفرض ترقية)
!pip install --quiet "unsloth[colab-new]"
print("✅ المكتبات مثبتة")
import torch, transformers, datasets, trl
print("torch:", torch.__version__, "| transformers:", transformers.__version__)
print("datasets:", datasets.__version__, "| trl:", trl.__version__)

## 2) تنزيل بيانات التدريب من مستودعك

In [ ]:
import json
import urllib.request
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(exist_ok=True)

with urllib.request.urlopen(f"https://huggingface.co/api/datasets/{DATASET_REPO}") as r:
    info = json.load(r)

downloaded = 0
for f in info.get("siblings", []):
    name = f.get("rfilename", "")
    if not name.endswith(".jsonl") or name.startswith("archive/"):
        continue
    hf_hub_download(repo_id=DATASET_REPO, repo_type="dataset", filename=name,
                    local_dir=str(DATA_DIR), token=HF_TOKEN)
    downloaded += 1

print(f"📥 نُزّل {downloaded} ملف JSONL")
jsonl_files = sorted(DATA_DIR.glob("*.jsonl"))
assert jsonl_files, "لا توجد ملفات JSONL في المستودع!"

## 3) تجهيز البيانات بتنسيق المحادثة

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "أنت مترجم محترف متخصص في المانجا والمانهوا، تنقل المعنى والروح "
    "إلى عربية فصحى ميسّرة. لا تترجم حرفياً؛ أعد الصياغة بأسلوب طبيعي "
    "مع الحفاظ على علامات التعجب والاستفهام وروح الحوار."
)

# أمان: لو لم تُشغَّل خلية التنزيل سابقاً، ننزّل الآن تلقائياً
if 'jsonl_files' not in globals() or not jsonl_files:
    print("⚠️ خلية التنزيل لم تُنفَّذ — ننزّل البيانات الآن تلقائياً...")
    import urllib.request
    from huggingface_hub import hf_hub_download
    DATA_DIR = Path("/content/data")
    DATA_DIR.mkdir(exist_ok=True)
    with urllib.request.urlopen(f"https://huggingface.co/api/datasets/{DATASET_REPO}") as r:
        info = json.load(r)
    downloaded = 0
    for f in info.get("siblings", []):
        name = f.get("rfilename", "")
        if not name.endswith(".jsonl") or name.startswith("archive/"):
            continue
        hf_hub_download(repo_id=DATASET_REPO, repo_type="dataset", filename=name,
                        local_dir=str(DATA_DIR), token=HF_TOKEN)
        downloaded += 1
    print(f"📥 نُزّل {downloaded} ملف JSONL")
    jsonl_files = sorted(DATA_DIR.glob("*.jsonl"))
    assert jsonl_files, "لا توجد ملفات JSONL في المستودع! تحقق من DATASET_REPO."
else:
    print(f"📂 نستخدم {len(jsonl_files)} ملف JSONL منزّلاً مسبقاً")

pairs = []
for path in jsonl_files:
    for line in path.open("r", encoding="utf-8"):
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except Exception:
            continue
        src = (rec.get("source") or "").strip()
        tgt = (rec.get("translation") or "").strip()
        if src and tgt:
            pairs.append((src, tgt))

print(f"📊 إجمالي الأزواج: {len(pairs)}")
assert len(pairs) > 10, "البيانات قليلة جداً — اجمع المزيد ثم أعد التشغيل."

def format_example(example):
    return {
        "text": f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
                f"<|im_start|>user\nترجم إلى العربية: {example['source']}<|im_end|>\n"
                f"<|im_start|>assistant\n{example['translation']}<|im_end|>\n",
    }

dataset = Dataset.from_list([{"source": s, "translation": t} for s, t in pairs])
dataset = dataset.map(format_example)
print("✅ البيانات جاهزة — نموذج:")
print(dataset[0]["text"][:300])

## 4) تحميل النموذج وتجهيز LoRA

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("✅ النموذج جاهز للتدريب")

## 5) التدريب

المدة التقريبية: **20–60 دقيقة** حسب حجم البيانات. لا تقلق إذا طال — شغّل واتركه.

**مهم:** لا تغلق التبويب أو تقطع الاتصال أثناء التدريب، وإلا ضاع كل شيء. (تدريب بأكثر من ساعة قد ينقطع بسبب مهلة Colab — للمجموعات الكبيرة رقِّ الحساب أو كرر الرفع.)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer.train()
print("🎉 اكتمل التدريب")

## 6) رفع النموذج إلى مستودعك

يُرفع النموذج المدرب إلى `OUTPUT_REPO` على Hugging Face — ثم يمكنك استخدامه مباشرة أو تحويله لصيغة GGUF (للعمل محلياً).

In [ ]:
from unsloth import FastLanguageModel

model.save_pretrained_merged(OUTPUT_REPO, tokenizer, save_method="merged_16bit")
model.push_to_hub_merged(OUTPUT_REPO, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
print(f"✅ النموذج في: https://huggingface.co/{OUTPUT_REPO}")

## 7) تجربة سريعة (اختياري)

جرّب ترجمة جملة بنفس النموذج المدرب قبل الخروج.

In [ ]:
FastLanguageModel.for_inference(model)
test_src = pairs[0][0] if pairs else "Hello"
prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\nترجم إلى العربية: {test_src}<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, top_p=0.95)
print("الأصل:", test_src)
print("الترجمة:", tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

---
### عند انتهائك

النموذج أصبح في `https://huggingface.co/{OUTPUT_REPO}`.

لاستخدامه:
- عبر **Hugging Face Inference** (مجاني مع حدود) من صفحة النموذج.
- أو محلياً عبر `transformers` / `llama.cpp` بعد تحويله لـ GGUF.